In [50]:
import pandas as pd

df1 = pd.read_csv("csvs/minhashblocksample_noblocks.ref.csv").rename(columns={"score": "noblocks_score"})
df2 = pd.read_csv("csvs/minhashblocksample_blocks.ref.csv").rename(columns={"score": "blocks_score"})
D = df1.merge(df2, on=["doc_id", "method"])
D["delta"] = D["blocks_score"] - D["noblocks_score"]

In [66]:
def load_dcpdd():
    dcpddblocks = pd.read_json("/Users/abha4861/dolma/dcpdd/output/metrics/minhashblocksample/dobolyilab/blockbench-blocksbin.jsonl", lines=True)
    dcpddblocks = dcpddblocks.rename(columns={"pred": "blocks_score", "id": "doc_id"})
    dcpddblocks["method"] = "dcpdd"

    dcpddnoblocks = pd.read_json("/Users/abha4861/dolma/dcpdd/output/metrics/minhashblocksample/dobolyilab/blockbench-noblocksbin.jsonl", lines=True)
    dcpddnoblocks = dcpddnoblocks.rename(columns={"pred": "noblocks_score", "id": "doc_id"})
    dcpddnoblocks['method'] = "dcpdd"
    return dcpddblocks, dcpddnoblocks

In [67]:
import pandas as pd

from scipy.stats import wilcoxon

from string import Template

for method in ["loss", "min_k", "ref-stablelm-base-alpha-3b-v2", "dcpdd"]:

    stats = pd.read_csv("stats.csv.gz")

    if method != "dcpdd":
        method2pattern = {"loss": Template(f"csvs/minhashblocksample_$kind.lite.csv"),
                          "min_k": Template(f"csvs/minhashblocksample_$kind.lite.csv"),
                          "ref-stablelm-base-alpha-3b-v2": Template(f"csvs/minhashblocksample_$kind.ref.csv")}

        noblocks = pd.read_csv(method2pattern[method].substitute({'kind': "noblocks"})).rename(columns={"score": "noblocks_score"})
        blocks = pd.read_csv(method2pattern[method].substitute({'kind': "blocks"})).rename(columns={"score": "blocks_score"})
    else:
        blocks, noblocks = load_dcpdd()
        
    noblocks = stats.merge(noblocks, left_on='url', right_on="doc_id").drop(columns=['size'])
    blocks = stats.merge(blocks, left_on='url', right_on="doc_id")

    D = noblocks.merge(blocks, on=["doc_id", "method"])
    D["delta"] = D["blocks_score"] - D["noblocks_score"]

    D = D[D["method"] == method].copy()
    if method == "dcpdd":
        D["delta"] *= -1

    D["size_bin"] = pd.cut(D["size"], bins=range(0, 50, 10), right=True)

    df = D.groupby("size_bin", observed=True).agg(
        delta_mean=("delta", "mean"),
        count=("delta", "count")
    ).reset_index()

    if method == "ref-stablelm-base-alpha-3b-v2":
        method = 'ref'
    
    df['method'] = method
    df.to_csv(method + ".csv", index=False)
    stat, p = wilcoxon(D["delta"].to_list(), alternative="greater")

    print(f"{D['delta'].mean():.3g}", method, p)
    
    import os
    os.system(f"cat {method}.csv")


0.0613 loss 2.617349818916068e-99
size_bin,delta_mean,count,method
"(0, 10]",0.1277935071413938,2760,loss
"(10, 20]",0.06261685170286238,4192,loss
"(20, 30]",0.019863255160410456,1516,loss
"(30, 40]",0.005161259170986464,1096,loss
0.173 min_k 4.009449276298088e-68
size_bin,delta_mean,count,method
"(0, 10]",0.3298557277929751,2760,min_k
"(10, 20]",0.18945783116096498,4192,min_k
"(20, 30]",0.06364951847282448,1516,min_k
"(30, 40]",0.02156545122175076,1096,min_k
0.065 ref 5.690380484910952e-98
size_bin,delta_mean,count,method
"(0, 10]",0.1277935071413938,2760,ref
"(10, 20]",0.06261685170286238,4192,ref
"(20, 30]",0.019863255160410466,1516,ref
"(30, 40]",0.005161259170986456,1096,ref
-0.00534 dcpdd 0.3260425978337291
size_bin,delta_mean,count,method
"(0, 10]",-0.06895390912849532,66655,dcpdd
"(10, 20]",0.2764674164114516,10738,dcpdd
"(20, 30]",0.1772120457946218,3938,dcpdd
"(30, 40]",0.11730163841149742,2667,dcpdd
